# M2 Joint N/V LightGCN v1.2 빠른 1차 실험

먼저 H&M 60일의 `M1` 대비 `joint_nv`만 실행합니다. Dunnhumby는 H&M 결과를 확인한 뒤 선택하도록 기본 비활성화하며, 두 데이터 모두 seed 42 validation만 사용합니다. test/holdout은 만들지 않습니다.

사용자 N축은 반복거래 횟수·거래 최근시점·관찰기간·평균 거래간격, V축은 거래당 평균금액을 사용합니다. 게이트는 `q_N=P(반복거래횟수/관찰기간)`, `q_V=P(거래당 평균금액)`로 고정합니다. 학습 강도 gamma의 제곱근을 사용자·아이템 N/V 블록 양쪽에 대칭 적용하고, Premium과 기존 `[F_p,T_p,R_p]/[AOV_p,Prem_p]`는 사용하지 않습니다. 별도 encoder·동결·후처리 없이 LightGCN 내부에서 plain BPR 하나로 공동학습합니다.

각 epoch의 진행 상태와 재개 checkpoint는 Drive의 결과 폴더에 저장됩니다. 연결이 끊긴 뒤 같은 노트북을 처음부터 다시 실행하면 마지막 저장 epoch에서 이어서 학습합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess

REVIEWED_SHA = '9095fe6c1bdbecf50223112e2c4c8b0d38ddbf62'
repo = Path('/content/clv-m2-lightgcn-runner')
os.chdir('/content')
if repo.exists():
    shutil.rmtree(repo)
subprocess.run(['git', 'clone', '-q', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(repo)], check=True)
os.chdir(repo)
subprocess.run(['git', 'checkout', '-q', REVIEWED_SHA], check=True)
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('code:', REVIEWED_SHA)

In [ ]:
import importlib, json, sys, torch
# 같은 Colab 세션에서 새 commit을 checkout해도 Python이 이전 repo 모듈을 재사용하지 않게 한다.
for module_name, module in list(sys.modules.items()):
    module_path = getattr(module, '__file__', '') or ''
    if module_path and str(repo) in str(module_path):
        del sys.modules[module_name]
importlib.invalidate_caches()
from IPython.display import display
import pandas as pd
from lightgcn_clv_joint_nv import (
    configure_joint_nv_run, preflight_summary, run_experiment
)
assert torch.cuda.is_available(), '런타임 > 런타임 유형 변경에서 GPU를 선택하세요.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
hm_cfg = configure_joint_nv_run('hm', short_hm=True)
dh_cfg = configure_joint_nv_run('dunnhumby', short_hm=False)
print('===== H&M 60일 실행 설정 =====')
print(json.dumps(preflight_summary(hm_cfg), ensure_ascii=False, indent=2))
print('===== Dunnhumby 전체 기간 실행 설정 =====')
print(json.dumps(preflight_summary(dh_cfg), ensure_ascii=False, indent=2))
print('review complete: 다음 셀을 실행하면 학습이 시작됩니다.')

In [ ]:
results = {}
print('===== H&M 60일 validation 시작 =====')
results['hm_w60'] = run_experiment(hm_cfg)
print('===== H&M 60일 완료: 결과를 먼저 확인하세요 =====')

In [ ]:
# H&M 60일 결과를 본 후 Dunnhumby도 돌릴 때만 True로 바꾸세요.
RUN_DUNNHUMBY = False
if RUN_DUNNHUMBY:
    print('===== Dunnhumby validation 시작 =====')
    results['dunnhumby_full'] = run_experiment(dh_cfg)
else:
    print('Dunnhumby는 실행하지 않았습니다.')

In [ ]:
for label, frame in results.items():
    print(f'\n===== {label}: M1 vs joint_nv =====')
    columns = [
        'model_id', 'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
        'recall@50', 'ndcg@50', 'revenue@10', 'arp@10', 'coverage@10',
        'n_distinct@10', 'exposure_entropy@10', 'eff_catalog@10',
        'top10_share@10', 'top100_share@10', 'value_alignment'
    ]
    display(frame[[c for c in columns if c in frame.columns]])
    print('판정:', frame.attrs['decision'])
    delta = pd.read_csv(frame.attrs['result_paths']['delta_csv'])
    display(delta)
    print('선수 train 내부 N/V 변수 타당성:')
    validity = pd.read_csv(frame.attrs['result_paths']['variable_validity_csv'])
    display(validity)
    print('N/V 사분면별 미래 거래:')
    quadrants = pd.read_csv(frame.attrs['result_paths']['variable_quadrants_csv'])
    display(quadrants)
    print('결과 파일:', frame.attrs['result_paths'])
print('완료. 위 표와 판정을 그대로 공유해 주세요.')